In [31]:
import zipfile
import os

from google.colab import drive

import os
import cv2
import numpy as np
import pandas as pd

from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt

from tqdm import trange


In [ ]:
class ImageDatasetExtractor:
    def __init__(self, zip_path, extract_to):
        self.zip_path = zip_path
        self.extract_to = extract_to

    def extract(self):
        print(f"Extracting {self.zip_path} to {self.extract_to}...")
        with zipfile.ZipFile(self.zip_path, 'r') as zip_ref:
            zip_ref.extractall(self.extract_to)
        print("Extraction complete.")


In [32]:
drive.mount('/content/drive')

# zip_path = '/content/drive/MyDrive/University/Bachelor/FinalProject/Image_Emotions.zip'
# extract_to = '/content/drive/MyDrive/University/Bachelor/FinalProject/image_emotions'

# extractor = ImageDatasetExtractor(zip_path, extract_to)
# extractor.extract()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
class ImagePreprocessor:
    def __init__(self, base_dir, target_size=(48, 48), color_mode='rgb'):
        self.base_dir = base_dir
        self.target_size = target_size
        self.color_mode = color_mode.lower()
        self.class_names = sorted(os.listdir(base_dir))
        self.class_to_index = {cls: idx for idx, cls in enumerate(self.class_names)}

    def load_images(self):
        X = []
        y = []

        for cls in self.class_names:
            cls_path = os.path.join(self.base_dir, cls)
            if not os.path.isdir(cls_path):
                continue

            for img_name in os.listdir(cls_path):
                img_path = os.path.join(cls_path, img_name)
                img = self.resize_image(img_path)
                if img is not None:
                    X.append(img)
                    y.append(self.class_to_index[cls])

        X = np.array(X, dtype=np.float32)
        y = np.array(y, dtype=np.int64)
        return X, y

    def resize_image(self, path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

        img = cv2.resize(img, self.target_size)
        img = img / 255.0

        img = np.expand_dims(img, axis=-1)

        return img


In [ ]:
train_dir = '/content/drive/MyDrive/University/Bachelor/FinalProject/image_emotions/train'
test_dir = '/content/drive/MyDrive/University/Bachelor/FinalProject/image_emotions/test'

train_preprocessor = ImagePreprocessor(train_dir, target_size=(48, 48), color_mode='grayscale')
X_train, y_train = train_preprocessor.load_images()

test_preprocessor = ImagePreprocessor(test_dir, target_size=(48, 48), color_mode='grayscale')
X_test, y_test = test_preprocessor.load_images()

print(f"Train shape: {X_train.shape}, Labels: {y_train.shape}")
print(f"Test shape: {X_test.shape}, Labels: {y_test.shape}")


Train shape: (28709, 48, 48, 1), Labels: (28709,)
Test shape: (7178, 48, 48, 1), Labels: (7178,)


In [ ]:
class CNNFeatureExtractor:
    def __init__(self, input_shape=(48,48,1)):
        self.model = self._build_model(input_shape)

    def _build_model(self, input_shape):
        model = models.Sequential([
            layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
            layers.MaxPooling2D(2,2),
            layers.Conv2D(64, (3,3), activation='relu'),
            layers.MaxPooling2D(2,2),
            layers.Flatten(),
            layers.Dense(128, activation='relu'),  # Feature layer
        ])
        return model

    def extract_features(self, images):
        # images shape: (num_samples, height, width, channels)
        features = self.model.predict(images, batch_size=32, verbose=1)
        return features

    def save_features_to_csv(self, features, labels, csv_path):
        num_features = features.shape[1]
        column_names = [f"feature_{i+1}" for i in range(num_features)] + ['label']
        df = pd.DataFrame(np.column_stack((features, labels)), columns=column_names)
        df.to_csv(csv_path, index=False)
        print(f"Features saved to {csv_path} with named columns")



In [ ]:
extractor = CNNFeatureExtractor(input_shape=(48,48,1))

features = extractor.extract_features(X_train)

extractor.save_features_to_csv(features, y_train, "/content/drive/MyDrive/University/Bachelor/FinalProject/Datasets/image_emotions.csv")


898/898 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Features saved to /content/drive/MyDrive/University/Bachelor/FinalProject/Datasets/image_emotions.csv with named columns


In [33]:
class EmotionClassifier:
    def __init__(self, csv_path):
        self.csv_path = csv_path
        self.label_encoder = LabelEncoder()
        self.model = None
        self.history = None
        self.X_test = None
        self.y_test = None

    def load_data(self, test_size=0.2):
        df = pd.read_csv(self.csv_path)
        X = df.drop('label', axis=1).values
        y = df['label'].values
        y_encoded = self.label_encoder.fit_transform(y)
        y_onehot = tf.keras.utils.to_categorical(y_encoded)
        X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=test_size, random_state=42)
        self.X_test = X_test
        self.y_test = y_test
        return X_train, X_test, y_train, y_test

    def build_model(self, input_dim, output_dim):
        self.model = models.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(32, activation='relu'),
            layers.Dropout(0.3),
            layers.Dense(output_dim, activation='softmax')
        ])
        self.model.compile(optimizer='adam',
                           loss='categorical_crossentropy',
                           metrics=['accuracy'])

    def train(self, X_train, y_train, epochs=30, batch_size=32, validation_split=0.2):
        for epoch in trange(1, epochs + 1, desc="Training Epochs"):
            history = self.model.fit(X_train, y_train,
                                     epochs=1,
                                     batch_size=batch_size,
                                     validation_split=validation_split,
                                     verbose=0)

            train_loss = history.history['loss'][0]
            train_acc = history.history['accuracy'][0]
            test_loss, test_acc = self.model.evaluate(self.X_test, self.y_test, verbose=0)

        print(f"Epoch {epoch:02d} | Train Acc : {train_acc:.4f}, Test Acc : {test_acc:.4f}, "
              f"Train Loss : {train_loss:.4f}, Test Loss : {test_loss:.4f}")

In [34]:
class ModelSaver:
    def __init__(self, model):
        self.model = model

    def save(self, path):
        if self.model:
            self.model.save(path)
            print(f"Model saved to {path}")
        else:
            print("No model found to save.")


In [ ]:
classifier = EmotionClassifier('/content/drive/MyDrive/University/Bachelor/FinalProject/Datasets/image_emotions.csv')
X_train, X_test, y_train, y_test = classifier.load_data()
classifier.build_model(input_dim=X_train.shape[1], output_dim=y_train.shape[1])
classifier.train(X_train, y_train, epochs=1000)


Training Epochs:  11%|█         | 106/1000 [04:53<39:09,  2.63s/it]

In [ ]:
saver = ModelSaver(classifier.model)
saver.save('/content/drive/MyDrive/University/Bachelor/FinalProject/Models/Image/emotion_model.h5')